# 06-P1. paper-aligned data selection

이 노트북은 paper-aligned Autoencoder 체인에 사용할 정상 행동 학습 구간과 event-wise 평가 구간을 고정한다.

핵심 역할:
- `normal_alignment` 기준 normal event window를 event id와 연결
- `fault_alignment` 기준 pre-fault window를 fault event와 연결
- event 단위 split을 다시 정의
- Autoencoder 학습용 normal window와 event evaluation용 window를 별도 저장


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for path in [current, *current.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('pyproject.toml을 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
ALIGNMENT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'label_alignment'
WINDOW_PATH = PROJECT_ROOT / 'data' / 'processed' / 'ml_windows' / 'ml_window_dataset.csv'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'paper_aligned'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NORMAL_TRAINING_OUTPUT = OUTPUT_DIR / 'normal_behaviour_training_windows.csv'
EVENT_EVAL_OUTPUT = OUTPUT_DIR / 'event_evaluation_windows.csv'
AUDIT_OUTPUT = OUTPUT_DIR / 'paper_aligned_data_selection_audit.csv'
METADATA_OUTPUT = OUTPUT_DIR / 'paper_aligned_data_selection_metadata.json'

FAULT_ALIGNMENT_PATH = ALIGNMENT_DIR / 'fault_alignment.csv'
NORMAL_ALIGNMENT_PATH = ALIGNMENT_DIR / 'normal_alignment.csv'
DISTURBANCE_ALIGNMENT_PATH = ALIGNMENT_DIR / 'disturbance_alignment.csv'

print(f'project root: {PROJECT_ROOT}')
print(f'window input: {WINDOW_PATH}')
print(f'output dir: {OUTPUT_DIR}')


project root: C:\Project3\HeatGrid_Agent
window input: C:\Project3\HeatGrid_Agent\data\processed\ml_windows\ml_window_dataset.csv
output dir: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned


## 선택 규칙

1. normal event는 `normal_alignment.csv`와 window overlap으로 `normal_event_id`를 복원한다.
2. fault event는 `fault_event_id`를 그대로 사용하되 split은 event 단위로 다시 계산한다.
3. normal behaviour 학습 후보는 `normal` 라벨만 사용한다.
4. Autoencoder 학습 제외 기준은 `data_quality_issue`, `use_for_supervised_training == False`, `normal_reference_outlier`, `maintenance_related`, `disturbance_count > 0`다.
5. event evaluation은 validation / holdout split만 사용하고, quality issue 여부는 별도 flag로 남긴다.
6. 현재 03번 window 체인을 재사용하므로 fault evaluation window 길이는 기존 생성 결과에 의존한다. 논문 기준 7일 고정 window는 이후 03/06 추가 보강 대상으로 남긴다.


In [2]:
DATETIME_COLUMNS = ['window_start', 'window_end']


def read_csv(path: Path, parse_dates=None) -> pd.DataFrame:
    return pd.read_csv(path, parse_dates=parse_dates)


def build_event_split_map(events: pd.DataFrame, *, sort_column: str) -> dict:
    split_map = {}
    for manufacturer, group in events.sort_values(['manufacturer', sort_column, 'event_id']).groupby('manufacturer'):
        event_ids = group['event_id'].tolist()
        event_count = len(event_ids)
        train_end = int(np.floor(event_count * 0.70))
        validation_end = int(np.floor(event_count * 0.85))
        for event_index, event_id in enumerate(event_ids):
            if event_index < train_end:
                split_map[int(event_id)] = 'train'
            elif event_index < validation_end:
                split_map[int(event_id)] = 'validation'
            else:
                split_map[int(event_id)] = 'holdout'
    return split_map


def bool_value(value) -> bool:
    if pd.isna(value):
        return False
    return bool(value)


def interval_match_event(row: pd.Series, event_table: pd.DataFrame) -> pd.Series:
    matches = event_table[
        event_table['window_start'].le(row['window_end'])
        & event_table['window_end'].ge(row['window_start'])
    ]
    if matches.empty:
        raise ValueError(
            f"normal event match 없음: manufacturer={row['manufacturer']} substation={row['substation_id']} window=({row['window_start']}, {row['window_end']})"
        )
    if len(matches) > 1:
        raise ValueError(
            f"normal event match 중복: manufacturer={row['manufacturer']} substation={row['substation_id']} window=({row['window_start']}, {row['window_end']})"
        )
    return matches.iloc[0]


def normal_exclusion_reason(row: pd.Series) -> str:
    reasons = []
    if bool_value(row.get('data_quality_issue')):
        reasons.append('data_quality_issue')
    if not bool_value(row.get('use_for_supervised_training', True)):
        reasons.append('use_for_supervised_training_false')
    if bool_value(row.get('normal_reference_outlier')):
        reasons.append('normal_reference_outlier')
    if bool_value(row.get('maintenance_related')):
        reasons.append('maintenance_related')
    if pd.notna(row.get('disturbance_count')) and float(row.get('disturbance_count', 0)) > 0:
        reasons.append('disturbance_overlap')
    return 'usable' if not reasons else '|'.join(reasons)


def eval_exclusion_reason(row: pd.Series) -> str:
    reasons = []
    if row['event_split'] == 'train':
        reasons.append('train_event')
    if bool_value(row.get('data_quality_issue')):
        reasons.append('data_quality_issue')
    return 'usable' if not reasons else '|'.join(reasons)


EVAL_WINDOW_DAYS = 7
EVAL_WINDOW_DURATION = pd.Timedelta(days=EVAL_WINDOW_DAYS)


def mark_eval_window(df: pd.DataFrame, *, reference_end_column: str) -> pd.DataFrame:
    result = df.copy()
    reference_start = result[reference_end_column] - EVAL_WINDOW_DURATION
    result['in_fixed_eval_window'] = result['window_end'].gt(reference_start) & result['window_end'].le(result[reference_end_column])
    result['eval_window_start'] = reference_start
    result['eval_window_end'] = result[reference_end_column]
    return result


windows = read_csv(WINDOW_PATH, parse_dates=DATETIME_COLUMNS)
fault_alignment = read_csv(
    FAULT_ALIGNMENT_PATH,
    parse_dates=['report_date', 'window_start', 'window_end', 'data_start', 'data_end', 'overlap_start', 'overlap_end'],
)
normal_alignment = read_csv(
    NORMAL_ALIGNMENT_PATH,
    parse_dates=['window_start', 'window_end', 'data_start', 'data_end', 'overlap_start', 'overlap_end'],
)
disturbance_alignment = read_csv(
    DISTURBANCE_ALIGNMENT_PATH,
    parse_dates=['event_start', 'window_start', 'window_end', 'data_start', 'data_end', 'overlap_start', 'overlap_end'],
)

fault_alignment = fault_alignment[fault_alignment['is_usable'] == True].copy()
normal_alignment = normal_alignment[normal_alignment['is_usable'] == True].copy()
disturbance_alignment = disturbance_alignment[disturbance_alignment['is_usable'] == True].copy()

fault_event_split_map = build_event_split_map(fault_alignment, sort_column='report_date')
normal_event_split_map = build_event_split_map(normal_alignment, sort_column='window_end')

normal_windows = windows[windows['label'].eq('normal')].copy()
normal_records = []

for (manufacturer, substation_id), group in normal_windows.groupby(['manufacturer', 'substation_id'], sort=False):
    event_table = normal_alignment[
        normal_alignment['manufacturer'].eq(manufacturer)
        & normal_alignment['substation_id'].eq(substation_id)
    ].copy()
    for _, row in group.iterrows():
        matched_event = interval_match_event(row, event_table)
        record = row.to_dict()
        record['event_type'] = 'normal_event'
        record['event_id'] = int(matched_event['event_id'])
        record['event_label'] = 'normal'
        record['event_start'] = matched_event['window_start']
        record['event_end'] = matched_event['window_end']
        record['report_date'] = pd.NaT
        record['event_split'] = normal_event_split_map[int(matched_event['event_id'])]
        record['selection_reason'] = normal_exclusion_reason(row)
        selected_base = record['selection_reason'] == 'usable'
        record['selected_for_autoencoder_train'] = selected_base and record['event_split'] == 'train'
        record['selected_for_autoencoder_validation'] = selected_base and record['event_split'] == 'validation'
        record['selected_for_normal_event_holdout'] = selected_base and record['event_split'] == 'holdout'
        normal_records.append(record)

normal_selected = pd.DataFrame(normal_records)

fault_windows = windows[windows['label'].eq('pre_fault')].copy()
fault_windows['fault_event_id'] = fault_windows['fault_event_id'].astype('Int64')
fault_meta = fault_alignment[[
    'event_id',
    'manufacturer',
    'substation_id',
    'problem',
    'report_date',
    'window_start',
    'window_end',
]].copy()
fault_meta = fault_meta.rename(columns={
    'event_id': 'fault_event_id',
    'window_start': 'event_start',
    'window_end': 'event_end',
})

fault_selected = fault_windows.merge(
    fault_meta,
    on=['fault_event_id', 'manufacturer', 'substation_id'],
    how='left',
    validate='many_to_one',
)
fault_selected['event_type'] = 'fault_event'
fault_selected['event_id'] = fault_selected['fault_event_id'].astype('Int64')
fault_selected['event_label'] = fault_selected['fault_label']
fault_selected['event_split'] = fault_selected['event_id'].map(fault_event_split_map)
fault_selected['selection_reason'] = fault_selected.apply(eval_exclusion_reason, axis=1)
fault_selected['selected_for_autoencoder_train'] = False
fault_selected['selected_for_autoencoder_validation'] = False
fault_selected['selected_for_normal_event_holdout'] = False

normal_training_windows = normal_selected.copy()
normal_training_windows['eval_window_start'] = pd.NaT
normal_training_windows['eval_window_end'] = pd.NaT
normal_training_windows['in_fixed_eval_window'] = pd.NA
normal_training_windows['selected_for_autoencoder_fit'] = (
    normal_training_windows['selected_for_autoencoder_train']
    | normal_training_windows['selected_for_autoencoder_validation']
)

event_eval_normal = mark_eval_window(normal_selected, reference_end_column='event_end')
event_eval_normal['selected_for_event_eval'] = event_eval_normal.apply(lambda row: eval_exclusion_reason(row) == 'usable', axis=1)
event_eval_normal['selected_for_event_eval'] = event_eval_normal['selected_for_event_eval'] & event_eval_normal['in_fixed_eval_window']
event_eval_normal['selected_for_event_tuning'] = event_eval_normal['selected_for_event_eval'] & event_eval_normal['event_split'].eq('validation')
event_eval_normal['selected_for_event_holdout'] = event_eval_normal['selected_for_event_eval'] & event_eval_normal['event_split'].eq('holdout')
event_eval_normal['evaluation_reason'] = event_eval_normal.apply(eval_exclusion_reason, axis=1)
event_eval_normal.loc[~event_eval_normal['in_fixed_eval_window'], 'evaluation_reason'] = event_eval_normal.loc[~event_eval_normal['in_fixed_eval_window'], 'evaluation_reason'].replace('usable', 'outside_fixed_eval_window')
event_eval_normal.loc[(~event_eval_normal['in_fixed_eval_window']) & (event_eval_normal['evaluation_reason'] != 'outside_fixed_eval_window'), 'evaluation_reason'] = event_eval_normal.loc[(~event_eval_normal['in_fixed_eval_window']) & (event_eval_normal['evaluation_reason'] != 'outside_fixed_eval_window'), 'evaluation_reason'] + '|outside_fixed_eval_window'

event_eval_fault = mark_eval_window(fault_selected, reference_end_column='report_date')
event_eval_fault['selected_for_event_eval'] = event_eval_fault['selection_reason'].eq('usable')
event_eval_fault['selected_for_event_eval'] = event_eval_fault['selected_for_event_eval'] & event_eval_fault['in_fixed_eval_window']
event_eval_fault['selected_for_event_tuning'] = event_eval_fault['selected_for_event_eval'] & event_eval_fault['event_split'].eq('validation')
event_eval_fault['selected_for_event_holdout'] = event_eval_fault['selected_for_event_eval'] & event_eval_fault['event_split'].eq('holdout')
event_eval_fault['evaluation_reason'] = event_eval_fault['selection_reason']
event_eval_fault.loc[~event_eval_fault['in_fixed_eval_window'], 'evaluation_reason'] = event_eval_fault.loc[~event_eval_fault['in_fixed_eval_window'], 'evaluation_reason'].replace('usable', 'outside_fixed_eval_window')
event_eval_fault.loc[(~event_eval_fault['in_fixed_eval_window']) & (event_eval_fault['evaluation_reason'] != 'outside_fixed_eval_window'), 'evaluation_reason'] = event_eval_fault.loc[(~event_eval_fault['in_fixed_eval_window']) & (event_eval_fault['evaluation_reason'] != 'outside_fixed_eval_window'), 'evaluation_reason'] + '|outside_fixed_eval_window'

shared_columns = [
    'manufacturer',
    'substation_id',
    'window_start',
    'window_end',
    'label',
    'fault_label',
    'fault_event_id',
    'estimated_lead_time_hours',
    'configuration_type',
    'window_source_type',
    'split_time_based',
    'split_substation_based',
    'use_for_supervised_training',
    'data_quality_issue',
    'normal_reference_outlier',
    'maintenance_related',
    'disturbance_count',
    'event_type',
    'event_id',
    'event_label',
    'event_start',
    'event_end',
    'eval_window_start',
    'eval_window_end',
    'in_fixed_eval_window',
    'report_date',
    'event_split',
    'selection_reason',
]

normal_training_output = normal_training_windows[shared_columns + [
    'selected_for_autoencoder_train',
    'selected_for_autoencoder_validation',
    'selected_for_normal_event_holdout',
    'selected_for_autoencoder_fit',
]].sort_values(['event_split', 'manufacturer', 'substation_id', 'window_start']).reset_index(drop=True)

event_eval_output = pd.concat(
    [
        event_eval_normal[shared_columns + ['selected_for_event_eval', 'selected_for_event_tuning', 'selected_for_event_holdout', 'evaluation_reason']],
        event_eval_fault[shared_columns + ['selected_for_event_eval', 'selected_for_event_tuning', 'selected_for_event_holdout', 'evaluation_reason']],
    ],
    ignore_index=True,
).sort_values(['event_type', 'event_split', 'manufacturer', 'substation_id', 'window_start']).reset_index(drop=True)

normal_training_output.to_csv(NORMAL_TRAINING_OUTPUT, index=False, encoding='utf-8-sig')
event_eval_output.to_csv(EVENT_EVAL_OUTPUT, index=False, encoding='utf-8-sig')

audit_rows = [
    {'dataset': 'normal_alignment', 'metric': 'usable_event_count', 'value': int(len(normal_alignment))},
    {'dataset': 'fault_alignment', 'metric': 'usable_event_count', 'value': int(len(fault_alignment))},
    {'dataset': 'disturbance_alignment', 'metric': 'usable_row_count', 'value': int(len(disturbance_alignment))},
    {'dataset': 'normal_training_windows', 'metric': 'row_count', 'value': int(len(normal_training_output))},
    {'dataset': 'normal_training_windows', 'metric': 'selected_for_autoencoder_fit', 'value': int(normal_training_output['selected_for_autoencoder_fit'].sum())},
    {'dataset': 'normal_training_windows', 'metric': 'selected_for_autoencoder_train', 'value': int(normal_training_output['selected_for_autoencoder_train'].sum())},
    {'dataset': 'normal_training_windows', 'metric': 'selected_for_autoencoder_validation', 'value': int(normal_training_output['selected_for_autoencoder_validation'].sum())},
    {'dataset': 'normal_training_windows', 'metric': 'selected_for_normal_event_holdout', 'value': int(normal_training_output['selected_for_normal_event_holdout'].sum())},
    {'dataset': 'event_evaluation_windows', 'metric': 'row_count', 'value': int(len(event_eval_output))},
    {'dataset': 'event_evaluation_windows', 'metric': 'selected_for_event_eval', 'value': int(event_eval_output['selected_for_event_eval'].sum())},
    {'dataset': 'event_evaluation_windows', 'metric': 'selected_for_event_tuning', 'value': int(event_eval_output['selected_for_event_tuning'].sum())},
    {'dataset': 'event_evaluation_windows', 'metric': 'selected_for_event_holdout', 'value': int(event_eval_output['selected_for_event_holdout'].sum())},
    {'dataset': 'event_evaluation_windows_normal', 'metric': 'selected_for_event_eval', 'value': int(event_eval_normal['selected_for_event_eval'].sum())},
    {'dataset': 'event_evaluation_windows_fault', 'metric': 'selected_for_event_eval', 'value': int(event_eval_fault['selected_for_event_eval'].sum())},
]

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(AUDIT_OUTPUT, index=False, encoding='utf-8-sig')

metadata = {
    'paper_aligned_version': '06_p1_event_selection_v1',
    'window_input_path': str(WINDOW_PATH),
    'fault_alignment_path': str(FAULT_ALIGNMENT_PATH),
    'normal_alignment_path': str(NORMAL_ALIGNMENT_PATH),
    'disturbance_alignment_path': str(DISTURBANCE_ALIGNMENT_PATH),
    'fault_event_split_policy': 'per manufacturer chronological split by report_date, 70/15/15',
    'normal_event_split_policy': 'per manufacturer chronological split by normal event window_end, 70/15/15',
    'normal_training_exclusion_rules': [
        'data_quality_issue',
        'use_for_supervised_training_false',
        'normal_reference_outlier',
        'maintenance_related',
        'disturbance_overlap',
    ],
    'event_eval_exclusion_rules': [
        'train_event',
        'data_quality_issue',
    ],
    'eval_window_days': EVAL_WINDOW_DAYS,
    'note': 'event evaluation is clipped to the last 7 days before report_date for fault events and the last 7 days before event_end for normal events; short source events keep only available windows inside that interval.',
}

METADATA_OUTPUT.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

display(audit_df)
display(normal_training_output[['event_split', 'selected_for_autoencoder_fit']].groupby('event_split').sum().reset_index())
display(event_eval_output.groupby(['event_type', 'event_split'])['selected_for_event_eval'].sum().reset_index())
print(f'saved: {NORMAL_TRAINING_OUTPUT}')
print(f'saved: {EVENT_EVAL_OUTPUT}')
print(f'saved: {AUDIT_OUTPUT}')
print(f'saved: {METADATA_OUTPUT}')


,dataset,metric,value
0,normal_alignment,usable_event_count,65
1,fault_alignment,usable_event_count,73
2,disturbance_alignment,usable_row_count,311
3,normal_training_windows,row_count,1818
4,normal_training_windows,selected_for_autoencoder_fit,1485
5,normal_training_windows,selected_for_autoencoder_train,1216
6,normal_training_windows,selected_for_autoencoder_validation,269
7,normal_training_windows,selected_for_normal_event_holdout,286
8,event_evaluation_windows,row_count,2633
9,event_evaluation_windows,selected_for_event_eval,854


,event_split,selected_for_autoencoder_fit
0,holdout,0
1,train,1216
2,validation,269


,event_type,event_split,selected_for_event_eval
0,fault_event,holdout,104
1,fault_event,train,0
2,fault_event,validation,166
3,normal_event,holdout,304
4,normal_event,train,0
5,normal_event,validation,280


saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\normal_behaviour_training_windows.csv
saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\event_evaluation_windows.csv
saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\paper_aligned_data_selection_audit.csv
saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\paper_aligned_data_selection_metadata.json


## 다음 단계

- `normal_behaviour_training_windows.csv`를 입력으로 06-P2 Autoencoder baseline을 학습한다.
- `event_evaluation_windows.csv`는 06-P3에서 validation / holdout event-wise detection 평가에 사용한다.
- 현재 fault evaluation window 길이는 03번 기존 window 체인 제약을 받으므로, 논문 기준 7일 고정 평가가 필요하면 03번 window 생성 규칙을 추가 보강해야 한다.
